In [1]:
# ============================================================
# MACHINE LEARNING FUNDAMENTALS
# Interview-Oriented Notes + Code
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_regression, make_classification
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    KFold,
    StratifiedKFold
)
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ============================================================
# 1. PARAMETERS vs HYPERPARAMETERS
# ============================================================

"""
PARAMETER
---------
Learned by the model during training.

Example:
Linear Regression:
    y = w1*x1 + w2*x2 + b

w1, w2 and b are PARAMETERS.

The model learns them from data.

HYPERPARAMETER
--------------
Chosen BEFORE training.

Examples:
    Decision Tree -> max_depth
    Random Forest -> n_estimators
    Ridge -> alpha
    KNN -> n_neighbors

Interview:
"Parameters are learned from training data,
while hyperparameters control how the learning
process/model is configured."
"""

X = np.array([[1], [2], [3], [4], [5]])
y = np.array([2, 4, 6, 8, 10])

model = LinearRegression()
model.fit(X, y)

print("Learned parameter - coefficient:", model.coef_)
print("Learned parameter - intercept:", model.intercept_)


# ============================================================
# 2. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

"""
WHY?

We need to know whether the model generalizes
to unseen data.

TRAIN
-----
Used to learn parameters.

VALIDATION
----------
Used to choose models/hyperparameters.

TEST
----
Used only for final unbiased evaluation.

Typical:

70% TRAIN
15% VALIDATION
15% TEST
"""

X, y = make_classification(
    n_samples=1000,
    n_features=10,
    random_state=42
)

# First separate test set
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

# Then create validation set
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.1765,  # approximately 15% of original
    random_state=42,
    stratify=y_temp
)

print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))


# ============================================================
# 3. OVERFITTING
# ============================================================

"""
OVERFITTING
-----------

Model learns training data TOO well.

It starts learning:
    - noise
    - random fluctuations
    - unnecessary details

Result:

Training performance -> very high
Test performance     -> poor

Example:

Decision Tree with unlimited depth
can memorize training examples.
"""

X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=5,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

# Simple tree
tree_small = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

# Very complex tree
tree_large = DecisionTreeClassifier(
    max_depth=None,
    random_state=42
)

tree_small.fit(X_train, y_train)
tree_large.fit(X_train, y_train)

print("Small tree train:",
      tree_small.score(X_train, y_train))

print("Small tree test:",
      tree_small.score(X_test, y_test))

print("Large tree train:",
      tree_large.score(X_train, y_train))

print("Large tree test:",
      tree_large.score(X_test, y_test))

"""
INTERVIEW IDEA:

If training accuracy = 100%
but test accuracy is much lower,

the model is likely overfitting.
"""


# ============================================================
# 4. UNDERFITTING
# ============================================================

"""
UNDERFITTING
------------

Model is too simple to capture the real relationship.

Training performance -> poor
Test performance     -> poor

Example:

Using a linear model for a highly nonlinear problem.
"""

from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# Non-linear data
X = np.linspace(-3, 3, 100).reshape(-1, 1)
y = X[:, 0] ** 3 + np.random.normal(0, 2, 100)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

linear_model = LinearRegression()

linear_model.fit(X_train, y_train)

print("Linear train R2:",
      linear_model.score(X_train, y_train))

print("Linear test R2:",
      linear_model.score(X_test, y_test))


# ============================================================
# 5. BIAS
# ============================================================

"""
BIAS
----

Bias = error caused by overly simplistic assumptions.

High bias:
    Model is too simple.

Usually:
    High bias -> underfitting

Example:
Trying to fit a straight line to a strongly
nonlinear relationship.
"""

"""
INTERVIEW:

High bias means the model has strong assumptions
and cannot capture the underlying relationship.
"""


# ============================================================
# 6. VARIANCE
# ============================================================

"""
VARIANCE
--------

Variance measures how sensitive the model is
to changes in training data.

High variance:
    Small change in training data
    -> large change in learned model.

Usually:
    High variance -> overfitting

Decision trees are a classic example.
"""

"""
MENTAL MODEL:

High Bias:
    Model doesn't learn enough.

High Variance:
    Model learns TOO much from training data.
"""


# ============================================================
# 7. BIAS-VARIANCE TRADEOFF
# ============================================================

"""
GOAL:

Find a model complexity where:

    Bias is reasonably low
    AND
    Variance is reasonably low

Conceptually:

Model Complexity
       ↑
       |
Bias   ↓
Variance ↑
       |
       └──────────────────>

Too simple:
    High Bias

Too complex:
    High Variance

Sweet spot:
    Good generalization
"""

# Demonstration using polynomial regression

from sklearn.metrics import mean_squared_error

np.random.seed(42)

X = np.linspace(-3, 3, 100).reshape(-1, 1)
y = X[:, 0] ** 2 + np.random.normal(0, 1, 100)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

for degree in [1, 2, 5, 15]:

    model = make_pipeline(
        PolynomialFeatures(degree),
        LinearRegression()
    )

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_error = mean_squared_error(y_train, train_pred)
    test_error = mean_squared_error(y_test, test_pred)

    print(
        f"Degree={degree}, "
        f"Train Error={train_error:.2f}, "
        f"Test Error={test_error:.2f}"
    )


# ============================================================
# 8. REGULARIZATION
# ============================================================

"""
WHY?

Regularization prevents the model from becoming
unnecessarily complex.

Instead of only minimizing:

    Loss

we minimize:

    Loss + Penalty

The penalty discourages large model weights.

Two major types:

L1 -> Lasso
L2 -> Ridge
"""


# ============================================================
# 9. L1 REGULARIZATION
# ============================================================

"""
L1:

Loss + alpha * SUM(|w|)

Important property:

Can force some coefficients exactly to ZERO.

Therefore:

L1 can perform feature selection.
"""

from sklearn.linear_model import Lasso

X = np.random.randn(100, 10)

# Make only first 3 features useful
true_weights = np.array([
    5, 3, 2, 0, 0, 0, 0, 0, 0, 0
])

y = X @ true_weights + np.random.randn(100)

lasso = Lasso(alpha=0.1)

lasso.fit(X, y)

print("Lasso coefficients:")
print(lasso.coef_)

"""
Interview:

L1 encourages sparsity and can make coefficients
exactly zero.
"""


# ============================================================
# 10. L2 REGULARIZATION
# ============================================================

"""
L2:

Loss + alpha * SUM(w²)

It penalizes large weights.

Usually:
    coefficients become smaller

but are generally not exactly zero.
"""

ridge = Ridge(alpha=1.0)

ridge.fit(X, y)

print("Ridge coefficients:")
print(ridge.coef_)

"""
L1:
    Feature selection / sparse solution

L2:
    Shrinks coefficients
    More stable when features are correlated
"""


# ============================================================
# 11. EFFECT OF REGULARIZATION
# ============================================================

alphas = [0, 0.01, 0.1, 1, 10, 100]

for alpha in alphas:

    if alpha == 0:
        model = LinearRegression()
    else:
        model = Ridge(alpha=alpha)

    model.fit(X, y)

    print(
        "alpha =", alpha,
        "coefficient magnitude =",
        np.linalg.norm(model.coef_)
    )

"""
Increasing alpha:

    stronger penalty
          ↓
    smaller weights
          ↓
    simpler model

Too much regularization:

    underfitting
"""


# ============================================================
# 12. CLASS IMBALANCE
# ============================================================

"""
Example:

1000 transactions

990 -> Normal
10  -> Fraud

A model predicting EVERYTHING as Normal gets:

Accuracy = 99%

But it detects ZERO fraud.

Therefore accuracy can be misleading.
"""

X, y = make_classification(
    n_samples=1000,
    weights=[0.99, 0.01],
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

model = LogisticRegression()

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("Recall:", recall_score(y_test, pred))
print("F1:", f1_score(y_test, pred))

"""
INTERVIEW:

For imbalanced classification, don't blindly rely
on accuracy.

Consider:
    Precision
    Recall
    F1
    PR-AUC
    ROC-AUC

depending on the business problem.
"""


# ============================================================
# 13. CROSS VALIDATION
# ============================================================

"""
WHY?

One train/test split can be lucky or unlucky.

Cross-validation repeatedly trains/evaluates
the model on different portions of the data.

K-FOLD:

Dataset
   ↓
+----+----+----+----+----+
| F1 | F2 | F3 | F4 | F5 |
+----+----+----+----+----+

Iteration 1:
Test = F1
Train = F2,F3,F4,F5

Iteration 2:
Test = F2
Train = F1,F3,F4,F5

...

Final score = average
"""

X, y = make_classification(
    n_samples=1000,
    n_features=10,
    random_state=42
)

model = LogisticRegression(max_iter=1000)

scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("Fold scores:", scores)
print("Mean CV score:", scores.mean())
print("Std:", scores.std())

"""
IMPORTANT:

CV gives us an estimate of how consistently
the model performs across different splits.
"""


# ============================================================
# 14. STRATIFIED CROSS VALIDATION
# ============================================================

"""
For classification, use StratifiedKFold.

It attempts to preserve the class distribution
in each fold.
"""

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    LogisticRegression(max_iter=1000),
    X,
    y,
    cv=skf,
    scoring="f1"
)

print("Stratified CV F1:", scores)
print("Mean:", scores.mean())


# ============================================================
# 15. DATA LEAKAGE ⭐⭐⭐
# ============================================================

"""
DATA LEAKAGE
------------

Information that should NOT be available to the model
during training accidentally enters the training process.

This causes artificially good validation/test results.

VERY IMPORTANT INTERVIEW TOPIC.
"""


# ============================================================
# 16. LEAKAGE THROUGH SCALING
# ============================================================

"""
WRONG:

scaler.fit_transform(entire_dataset)

then

train_test_split()

Why?

The scaler has already seen test data.

Correct:

1. Split
2. Fit scaler ONLY on train
3. Transform train and test
"""

X, y = make_classification(
    n_samples=1000,
    n_features=10,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

scaler = StandardScaler()

# Correct
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Correct preprocessing completed.")


# ============================================================
# 17. PIPELINE PREVENTS PREPROCESSING LEAKAGE
# ============================================================

"""
Pipeline:

Scaler
   ↓
Model

During cross-validation:

Fold 1:
    scaler fits only on training portion
    model trains
    validation portion remains unseen

Fold 2:
    new scaler fitted on that fold's training data

etc.

This is why Pipeline is extremely important.
"""

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

scores = cross_val_score(
    pipeline,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("Pipeline CV scores:", scores)
print("Mean:", scores.mean())


# ============================================================
# 18. TARGET LEAKAGE
# ============================================================

"""
Target leakage:

A feature accidentally contains information
that is only known AFTER the target event.

Example:

Predict:
    "Will customer default?"

Bad feature:
    "Recovery amount after default"

That information only exists AFTER default.

The model can appear extremely accurate,
but cannot be used in real life.
"""


# ============================================================
# 19. DATA DUPLICATION
# ============================================================

"""
Duplicate records can also cause leakage.

Example:

Same customer appears in both:

TRAIN
and
TEST

The model may effectively see the same example twice.

Therefore data cleaning should happen carefully
before modeling.
"""


# ============================================================
# 20. FEATURE / TARGET SEPARATION
# ============================================================

"""
Always clearly separate:

X = features
y = target

Example:

X = df.drop("target", axis=1)
y = df["target"]
"""

df = pd.DataFrame({
    "age": [20, 30, 40, 50],
    "salary": [30000, 50000, 70000, 90000],
    "target": [0, 0, 1, 1]
})

X = df.drop("target", axis=1)
y = df["target"]

print("X:")
print(X)

print("y:")
print(y)


# ============================================================
# 21. RANDOM SEED / REPRODUCIBILITY
# ============================================================

"""
Many ML algorithms contain randomness.

random_state makes experiments reproducible.
"""

model1 = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model2 = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model1.fit(X, y)
model2.fit(X, y)

print(
    np.array_equal(
        model1.predict(X),
        model2.predict(X)
    )
)

"""
Interview:

random_state does NOT make the model better.

It makes the experiment reproducible.
"""


# ============================================================
# 22. FEATURE SCALING
# ============================================================

"""
Some algorithms are sensitive to feature scale.

Example:

Age:
    20 - 60

Salary:
    20,000 - 200,000

Distance-based algorithms can be dominated
by salary.

Scaling puts features on comparable scales.
"""

X = np.array([
    [20, 20000],
    [30, 50000],
    [40, 100000],
    [50, 200000]
])

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("Original:")
print(X)

print("\nScaled:")
print(X_scaled)

"""
Important:

Scaling is particularly important for:

    KNN
    K-Means
    SVM
    Logistic Regression
    Linear Regression in some optimization settings
    PCA
    Neural Networks

Tree-based models generally do NOT require scaling.
"""


# ============================================================
# 23. RANDOM VS STRATIFIED SPLIT
# ============================================================

"""
For classification, stratify=y helps preserve
class proportions.
"""

X, y = make_classification(
    n_samples=1000,
    weights=[0.9, 0.1],
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Overall positive ratio:", y.mean())
print("Train positive ratio:", y_train.mean())
print("Test positive ratio:", y_test.mean())


# ============================================================
# 24. MODEL COMPLEXITY
# ============================================================

"""
Model complexity controls how flexible the model is.

Decision Tree:

max_depth small
    ↓
simpler model
    ↓
higher bias

max_depth large
    ↓
more complex model
    ↓
higher variance
"""

for depth in [1, 2, 3, 5, 10, None]:

    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )

    scores = cross_val_score(
        model,
        X,
        y,
        cv=5
    )

    print(
        "Depth:", depth,
        "CV:", scores.mean()
    )


# ============================================================
# 25. TRAINING ERROR VS GENERALIZATION ERROR
# ============================================================

"""
Training error:

Error on data used to train the model.

Generalization error:

Error on unseen data.

The actual goal of ML is NOT:

    minimize training error

The goal is:

    perform well on unseen data.
"""

"""
Interview statement:

"Machine learning is fundamentally about
generalization, not memorization."
"""


# ============================================================
# 26. BASELINE MODEL
# ============================================================

"""
Always establish a baseline.

Example classification:

Predict majority class.

Then compare your ML model against it.

If ML model doesn't beat the baseline,
something is wrong or the problem may not
contain enough predictive information.
"""

from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(
    strategy="most_frequent"
)

dummy.fit(X_train, y_train)

baseline_accuracy = dummy.score(
    X_test,
    y_test
)

print("Baseline accuracy:", baseline_accuracy)


# ============================================================
# 27. LEARNING CURVE CONCEPT
# ============================================================

"""
Learning curve:

Training performance and validation performance
as training data increases.

Typical patterns:

High bias:
    train score low
    validation score low

High variance:
    train score high
    validation score much lower

This helps diagnose whether more data
might help.
"""

from sklearn.model_selection import learning_curve

model = DecisionTreeClassifier(
    max_depth=None,
    random_state=42
)

train_sizes, train_scores, validation_scores = learning_curve(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy",
    train_sizes=np.linspace(0.1, 1.0, 5)
)

print("Training scores:")
print(train_scores.mean(axis=1))

print("Validation scores:")
print(validation_scores.mean(axis=1))


# ============================================================
# 28. ERROR ANALYSIS
# ============================================================

"""
After training a model:

DON'T immediately stop at:

accuracy = 92%

Instead inspect:

    Which examples are wrong?
    Which class is difficult?
    Are there specific segments?
    Are errors concentrated in certain ranges?

This is called ERROR ANALYSIS.
"""

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

pred = model.predict(X_test)

errors = X_test[pred != y_test]

print("Number of errors:", len(errors))


# ============================================================
# 29. COMPLETE BASIC ML WORKFLOW
# ============================================================

"""
INTERVIEW-READY WORKFLOW:

1. Understand business problem
2. Define target
3. Collect data
4. Clean data
5. Check missing values
6. Check duplicates
7. Check class imbalance
8. Split train/test
9. Perform preprocessing
10. Establish baseline
11. Train simple model
12. Evaluate
13. Cross-validation
14. Tune hyperparameters
15. Evaluate final model
16. Perform error analysis
17. Explain model
18. Deploy
19. Monitor
"""

# Simple implementation

X, y = make_classification(
    n_samples=1000,
    n_features=20,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000
    ))
])

pipeline.fit(X_train, y_train)

pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("Recall:", recall_score(y_test, pred))
print("F1:", f1_score(y_test, pred))


# ============================================================
# 30. INTERVIEW CHEAT SHEET
# ============================================================

"""
Q: What is overfitting?

A:
Model performs very well on training data
but poorly on unseen data because it learned
noise/specific patterns from training data.

------------------------------------------------------------

Q: What is underfitting?

A:
Model is too simple to capture the underlying
relationship, so both training and test performance
are poor.

------------------------------------------------------------

Q: What is bias?

A:
Error from overly simplistic assumptions.

High bias -> underfitting.

------------------------------------------------------------

Q: What is variance?

A:
Sensitivity of the model to the particular
training dataset.

High variance -> overfitting.

------------------------------------------------------------

Q: How do you reduce overfitting?

A:
- More training data
- Regularization
- Simpler model
- Cross-validation
- Feature selection
- Early stopping
- Dropout for neural networks
- Pruning for trees

------------------------------------------------------------

Q: L1 vs L2?

L1:
    alpha * SUM(|w|)
    Can make coefficients exactly zero.

L2:
    alpha * SUM(w²)
    Shrinks coefficients.

------------------------------------------------------------

Q: Why cross-validation?

A:
To obtain a more reliable estimate of model
performance across different train/validation splits.

------------------------------------------------------------

Q: What is data leakage?

A:
When information that should not be available
during training enters the training process.

It produces overly optimistic evaluation.

------------------------------------------------------------

Q: Why use Pipeline?

A:
It ensures preprocessing is fitted only on the
training portion during cross-validation and
helps prevent leakage.

------------------------------------------------------------

Q: Why doesn't accuracy work well for imbalanced data?

A:
Because a model can achieve high accuracy by
predicting the majority class while performing
poorly on the minority class.

------------------------------------------------------------

Q: Why scale features?

A:
Algorithms based on distances or gradient/weight
optimization can be affected by feature magnitude.

------------------------------------------------------------

Q: Which algorithms usually don't require scaling?

A:
Tree-based algorithms such as:

Decision Tree
Random Forest
XGBoost
LightGBM
CatBoost

------------------------------------------------------------

Q: Parameter vs hyperparameter?

Parameter:
    Learned from data.

Hyperparameter:
    Set before/during training to control the
    learning/model configuration.

------------------------------------------------------------

Q: What is generalization?

A:
The ability of a model to perform well on
previously unseen data.

------------------------------------------------------------

Q: Why establish a baseline?

A:
To determine whether the ML model actually provides
useful predictive performance compared with a simple
strategy.

------------------------------------------------------------

Q: What is the main goal of ML?

A:
Generalization to unseen data.
"""

Learned parameter - coefficient: [2.]
Learned parameter - intercept: 0.0
Train: 699
Validation: 151
Test: 150
Small tree train: 0.9028571428571428
Small tree test: 0.88
Large tree train: 1.0
Large tree test: 0.8866666666666667
Linear train R2: 0.8017692313698294
Linear test R2: 0.761416280687915
Degree=1, Train Error=9.22, Test Error=7.28
Degree=2, Train Error=0.92, Test Error=0.56
Degree=5, Train Error=0.86, Test Error=0.56
Degree=15, Train Error=0.72, Test Error=0.88
Lasso coefficients:
[ 4.92086739  2.85339818  1.82295733 -0.         -0.          0.
 -0.         -0.0099685   0.         -0.07124164]
Ridge coefficients:
[ 4.95236237  2.92546424  1.92526602 -0.08329288 -0.12267129  0.09689533
 -0.03084775 -0.10723248  0.06364358 -0.16071098]
alpha = 0 coefficient magnitude = 6.143639036159858
alpha = 0.01 coefficient magnitude = 6.142909351066439
alpha = 0.1 coefficient magnitude = 6.136350941428294
alpha = 1 coefficient magnitude = 6.0716214360415
alpha = 10 coefficient magnitude = 5.

/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value 

Depth: 1 CV: 0.9289999999999999
Depth: 2 CV: 0.938
Depth: 3 CV: 0.966
Depth: 5 CV: 0.9600000000000002
Depth: 10 CV: 0.95
Depth: None CV: 0.9499999999999998
Baseline accuracy: 0.895
Training scores:
[1. 1. 1. 1. 1.]
Validation scores:
[0.922 0.924 0.946 0.932 0.95 ]
Number of errors: 18
Accuracy: 0.885
Precision: 0.8888888888888888
Recall: 0.88
F1: 0.8844221105527639


/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features

"\nQ: What is overfitting?\n\nA:\nModel performs very well on training data\nbut poorly on unseen data because it learned\nnoise/specific patterns from training data.\n\n------------------------------------------------------------\n\nQ: What is underfitting?\n\nA:\nModel is too simple to capture the underlying\nrelationship, so both training and test performance\nare poor.\n\n------------------------------------------------------------\n\nQ: What is bias?\n\nA:\nError from overly simplistic assumptions.\n\nHigh bias -> underfitting.\n\n------------------------------------------------------------\n\nQ: What is variance?\n\nA:\nSensitivity of the model to the particular\ntraining dataset.\n\nHigh variance -> overfitting.\n\n------------------------------------------------------------\n\nQ: How do you reduce overfitting?\n\nA:\n- More training data\n- Regularization\n- Simpler model\n- Cross-validation\n- Feature selection\n- Early stopping\n- Dropout for neural networks\n- Pruning for tr